# ETL de los Datos Climatológicos

Este *notebook* accede a los datos de SiAR del MAPA a través de una API. Dicha API se adjunta en este código, pero su uso queda restringido, pudiendo ser utilizada solo con fines correctivos por parte del tutor.

In [1]:
# Importar librerías
import os, sys, requests
import time as time_module
from datetime import datetime, date, timedelta, time
import numpy as np
import pandas as pd

In [2]:
def fetch_siar_hourly_limited(api_key, station_id, start_datetime, end_datetime, intermediate_csv):
    """
    Obtiene datos horarios de la API SiAR con gestión de límites y fragmentación automática.
    Realiza peticiones cada 2 días respetando límites de la API (240 peticiones/día, 10.000 registros)
    con pausas entre peticiones y guardado automático en caso de límite o error.
    
    Args:
        - api_key (str): Clave de acceso a la API SiAR
        - station_id (str): Código de estación (ej: "A09")  
        - start_datetime (str): Fecha inicio en formato "YYYY-MM-DD"
        - end_datetime (str): Fecha fin en formato "YYYY-MM-DD"
        - intermediate_csv (str): Ruta donde guardar datos si se alcanza límite o hay error
    
    Saves:
        - pd.DataFrame: DataFrame con datos horarios ordenados por fecha y hora,
                    o DataFrame vacío si no se obtuvieron datos
    
    Note:
        - Fragmenta en ventanas de 2 días con pausa de 65s entre peticiones
        - Guarda automáticamente datos parciales al alcanzar límites diarios
        - Convierte columna "Fecha" a datetime y ordena por ["Fecha", "HoraMin"]
    """
    # Parsear fecha-hora de inicio y fin
    dt_start = datetime.strptime(start_datetime, "%Y-%m-%d").date()
    dt_end = datetime.strptime(end_datetime, "%Y-%m-%d").date()
    if dt_end < dt_start:
        raise ValueError("end_datetime debe ser >= start_datetime")

    all_records = []
    base_url = "https://servicio.mapama.gob.es/apisiar/API/v1/Datos/Horarios/Estacion"

    # Contadores de peticiones
    requests_today = 0
    batch_counter = 0
    MAX_DAILY_REQUESTS = 240
    BATCH_SIZE = 96   # Tamaño del lote
    MAX_DAILY_BATCH = 10_000
    BATCH_PAUSE = 65  # Segundos a esperar tras cada BATCH_SIZE peticiones

    # Construir lista de ventanas diarias
    windows = []
    current_start = dt_start
    while current_start <= dt_end:
        current_end = current_start + timedelta(days=1)
        current_end = min(current_end, dt_end)
        windows.append((current_start, current_end))
        current_start += timedelta(days=2)

    try:
        for window_start, window_end in windows:
            # Comprobar límite diario
            if requests_today >= MAX_DAILY_REQUESTS or batch_counter >= MAX_DAILY_BATCH:
                # Guardar intermedio y terminar bucle
                if all_records:
                    df_partial = pd.DataFrame(all_records)
                    if "Fecha" in df_partial.columns:
                        df_partial["Fecha"] = pd.to_datetime(df_partial["Fecha"])
                    df_partial.sort_values(by=["Fecha", "HoraMin"], inplace=True)
                    df_partial.to_csv(intermediate_csv, index=False)
                    if requests_today >= MAX_DAILY_REQUESTS:
                        print(f"[INFO] Límite diario ({MAX_DAILY_REQUESTS} peticiones) alcanzado. Datos guardados en {intermediate_csv}.")
                    else:
                        print(f"[INFO] Límite diario ({MAX_DAILY_BATCH} registros) alcanzado. Datos guardados en {intermediate_csv}.")
                else:
                    print(f"[INFO] Límite diario alcanzado sin registros recogidos.")
                break

            # Preparar parámetros: la API devuelve horas completas del día designado
            params = {
                "ClaveAPI": api_key,
                "Id": station_id,
                "FechaInicial": window_start.strftime("%Y-%m-%d"),
                "FechaFinal": window_end.strftime("%Y-%m-%d")
            }

            # Realizar la petición
            resp = requests.get(base_url, params=params)
            requests_today += 1

            # Procesar estado
            if resp.status_code != 200:
                raise RuntimeError(
                    f"Error en petición [{window_start} a {window_end}]: {resp.status_code} - {resp.text}"
                )
            data = resp.json()
            datos = data.get("Datos")
            if not isinstance(datos, list): # Si datos no es una lista, entonces no contiene registros válidos
                mensaje = data.get("MensajeError") or data.get("MensajeRespuesta") or "Respuesta inesperada"
                raise RuntimeError(f"API devolvió error en ventana {window_start} a {window_end}: {mensaje}")

            # Añadir registros
            all_records.extend(datos)

            # Tras cada petición, esperar BATCH_PAUSE segundos
            print(f"[INFO] {BATCH_SIZE} registros obtenidos. Esperando {BATCH_PAUSE} s...")
            time_module.sleep(BATCH_PAUSE)
            batch_counter += BATCH_SIZE

    except Exception as e:
        # Guardar datos recogidos hasta el momento antes de propagar error
        if all_records:
            try:
                df_partial = pd.DataFrame(all_records)
                if ("Fecha" in df_partial.columns) and ("HoraMin" in df_partial.columns):
                    df_partial["Fecha"] = pd.to_datetime(df_partial["Fecha"])
                    df_partial.sort_values(by=["Fecha", "HoraMin"], inplace=True)
                df_partial.to_csv(intermediate_csv, index=False)
                print(f"[ERROR] Excepción: {e}. Datos parciales guardados en {intermediate_csv}.")
            except Exception as save_exc:
                print(f"[ERROR] No se pudo guardar datos parciales: {save_exc}")
        else:
            try:
                pd.DataFrame().to_csv(intermediate_csv, index=False)
                print(f"[ERROR] No había registros. CSV vacío guardado en {intermediate_csv}.")
            except Exception as save_exc:
                print(f"[ERROR] No se pudo guardar CSV vacío: {save_exc}")
        # Volver a lanzar la excepción
        raise

    # Construir DataFrame final si completó sin exceder límite diario
    if all_records:
        df = pd.DataFrame(all_records)
        if ("Fecha" in df.columns) and ("HoraMin" in df.columns):
            df["Fecha"] = pd.to_datetime(df["Fecha"])
            df = df.sort_values(by=["Fecha", "HoraMin"]).reset_index(drop=True)
        df.to_csv(intermediate_csv, index=False)
    else:
        pd.DataFrame().to_csv(intermediate_csv, index=False)
        print(f"[ERROR] No había registros. CSV vacío guardado en {intermediate_csv}.")

Debido a las limitaciones de la API de SiAR, solo se pueden obtener 10000 registros cada 24h. Eso conlleva que el siguiente bloque de código deberá ejecutarse 3 veces, en 3 días diferentes, alternando en cada ejecución la fecha de inicio `start_datetime` con las que aparecen comentadas. Es muy importante actualizar `iter` cada vez que se ejecute el bloque.

In [3]:
# Parámetros de entrada
api_key = "_J5ZHNJchBofibZEYRPDJKEFDuSGqdBJMvFpm5GhH78C53NiAD"
station_id = "A09"
start_datetime = "2023-12-31"      # "2023-12-31"   # "2024-07-14"   # "2025-02-08"
end_datetime = "2025-05-25"
iter = 1                    # Número de la iteración (OBLIGATORIO actualizar en cada iteración)
weather_path = "C:\\Users\\David\\Desktop\\Master\\Modulo TFM\\Notebooks modulares\\datos metereologicos"

# Llamada a la función (guarda cada dataset automáticamente)
fetch_siar_hourly_limited(
    api_key=api_key,
    station_id=station_id,
    start_datetime=start_datetime,
    end_datetime=end_datetime,
    intermediate_csv=os.path.join(weather_path, f"weather_hourly_incompleted_part{iter}.csv") # Obligatorio actualizar "iter"
)

[INFO] 96 registros obtenidos. Esperando 65 s...


La siguiente parte del código debe ejecutarse únicamente una vez se posean los 3 *datasets*. Se hace cambiando `full_data` a True.

In [ ]:
full_data = False

In [4]:
if full_data:
    store_path = "C:\\Users\\David\\Desktop\\Master\\Modulo TFM\\Notebooks modulares\\raw data"
    
    # Obtener todos los archivos que empiecen con "weather_hourly_incompleted" y terminen en ".csv"
    csv_files = sorted([
        f for f in os.listdir(weather_path)
        if f.startswith("weather_hourly_incompleted") and f.endswith(".csv")
    ])
    
    # Leer y combinar los csv
    dfs = []
    for file in csv_files:
        df = pd.read_csv(os.path.join(weather_path, file), parse_dates=["Fecha"])
        dfs.append(df)
    
    # Combinar y ordenar
    df_weather_hourly = (
        pd.concat(dfs)
        .drop_duplicates(subset=["Fecha", "HoraMin"]) 
        .sort_values(by=["Fecha", "HoraMin"])
        .reset_index(drop=True)
    )
    
    # Guardar csv
    df_weather_hourly.to_csv(os.path.join(store_path, "weather_hourly.csv"), index=False)